# 03 Encoder, prompt classification, and fine-tuning

This notebook contains the CPU-friendly advanced NLP experiments. The encoder section is intended to run locally. Prompt classification and fine-tuning are guarded by flags because they download models and can be slow on CPU.

In [ ]:
from pathlib import Path
import sys

_PROJECT_MARKERS = (
    Path('src') / 'data' / 'prepare_dataset.py',
    Path('Turismy') / 'reviews.csv',
)


def _is_project_root(path):
    return all((path / marker).exists() for marker in _PROJECT_MARKERS)


def _candidate_roots(start):
    start = start.resolve()
    seen = set()

    for candidate in (start, *start.parents):
        if candidate not in seen:
            seen.add(candidate)
            yield candidate

    for base in (start, start.parent):
        try:
            children = list(base.iterdir())
        except OSError:
            continue

        for child in children:
            try:
                if not child.is_dir():
                    continue
                candidate = child.resolve()
            except OSError:
                continue

            if candidate not in seen:
                seen.add(candidate)
                yield candidate


def find_project_root():
    for candidate in _candidate_roots(Path.cwd()):
        if _is_project_root(candidate):
            return candidate

    raise RuntimeError(
        'Project root was not found. Start the notebook from the repository root, '
        'the notebooks folder, or the parent folder that contains MasinskoUcenje-Projekat.'
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier

from src.data.prepare_dataset import LABEL_COLUMNS, prepare_dataset

dataset_path = PROJECT_ROOT / 'Turismy' / 'reviews_enriched.csv'
df = pd.read_csv(dataset_path) if dataset_path.exists() else prepare_dataset(output_path=dataset_path)

## Encoder classification

Sentence embeddings are generated with a small local `sentence-transformers` encoder, then a classical multi-label classifier is trained on top of those embeddings.

In [ ]:
RUN_ENCODER_EXPERIMENT = False

if RUN_ENCODER_EXPERIMENT:
    from sentence_transformers import SentenceTransformer

    sample = df.sample(n=min(2000, len(df)), random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        sample['clean_comments'], sample[list(LABEL_COLUMNS)], test_size=0.2, random_state=42
    )
    encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    X_train_embeddings = encoder.encode(X_train.tolist(), show_progress_bar=True)
    X_test_embeddings = encoder.encode(X_test.tolist(), show_progress_bar=True)
    classifier = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight='balanced'))
    classifier.fit(X_train_embeddings, y_train)
    predictions = classifier.predict(X_test_embeddings)
    print('Encoder micro F1:', f1_score(y_test, predictions, average='micro', zero_division=0))
else:
    print('Set RUN_ENCODER_EXPERIMENT = True to download the encoder model and run this CPU experiment.')

## Prompt-based generative classification

This section uses a small local text-to-text model. It is intentionally limited to a few examples because CPU inference can be slow.

In [ ]:
RUN_PROMPT_EXPERIMENT = False

if RUN_PROMPT_EXPERIMENT:
    from transformers import pipeline

    generator = pipeline('text2text-generation', model='google/flan-t5-small')
    comment = df['clean_comments'].iloc[0]
    prompt = (
        'Classify this review with 0 or 1 for cleanliness, location, luxury, and family_friendly. '
        'Return JSON only. Review: ' + comment
    )
    print(generator(prompt, max_new_tokens=80)[0]['generated_text'])
else:
    print('Set RUN_PROMPT_EXPERIMENT = True to run local prompt-based classification.')

## Fine-tuning note

For CPU-only execution, fine-tuning should be demonstrated on a small subset with a small transformer and few epochs. The report should explain that this experiment is constrained by hardware and is not expected to dominate the classical TF-IDF baseline.